# 07 — Optimisation des performances et profiling

Ce notebook évalue les performances du système de scoring crédit déployé.

Objectifs :
- mesurer le temps d’inférence local du pipeline ;
- mesurer la latence bout-en-bout de l’API FastAPI ;
- comparer l’inférence unitaire et l’inférence batch ;
- identifier les goulots d’étranglement avec cProfile ;
- tester une optimisation avec ONNX Runtime.

## Chargement des résultats de benchmark

Les benchmarks sont générés par les scripts :

- `scripts/profile_inference.py`
- `scripts/benchmark_api.py`
- `scripts/benchmark_batching.py`
- `scripts/benchmark_onnx.py`

Les résultats sont sauvegardés dans `reports/performance/`.

In [1]:
import json

with open("../reports/performance/inference_benchmark.json") as f:
    inference = json.load(f)

with open("../reports/performance/api_benchmark.json") as f:
    api = json.load(f)

with open("../reports/performance/batching_benchmark.json") as f:
    batching = json.load(f)

inference, api, batching

({'n_rows': 200,
  'n_loops': 20,
  'avg_ms': 11.654050000197458,
  'min_ms': 10.858900001039729,
  'max_ms': 18.788399998811656,
  'stats_file': 'reports\\performance\\cprofile_inference.prof',
  'top20_file': 'reports\\performance\\cprofile_inference_top20.txt'},
 {'n_requests': 50,
  'avg_ms': 16.321420000058424,
  'median_ms': 14.43699999981618,
  'min_ms': 12.50369999979739,
  'max_ms': 100.83730000042124,
  'status_codes': {'200': 50}},
 {'single_total_ms_for_50_calls': 146.37039999979606,
  'single_avg_ms_per_row': 2.9274079999959213,
  'batch_total_ms_for_50_rows': 4.449999998541898,
  'batch_avg_ms_per_row': 0.08899999997083796,
  'speedup_factor_per_row': 32.89222472983287})

## Performance locale du pipeline

Nous mesurons ici le temps d’exécution du pipeline complet scikit-learn.

Ce temps inclut :
- le prétraitement ;
- l’imputation ;
- la transformation des données ;
- l’inférence LightGBM.

In [2]:
inference

{'n_rows': 200,
 'n_loops': 20,
 'avg_ms': 11.654050000197458,
 'min_ms': 10.858900001039729,
 'max_ms': 18.788399998811656,
 'stats_file': 'reports\\performance\\cprofile_inference.prof',
 'top20_file': 'reports\\performance\\cprofile_inference_top20.txt'}

## Performance bout-en-bout de l’API

Nous mesurons ici la latence complète d’une requête API.

Cette latence inclut :
- l’envoi HTTP ;
- la validation du payload ;
- la sérialisation / désérialisation JSON ;
- l’inférence modèle ;
- la génération de la réponse.

In [3]:
api

{'n_requests': 50,
 'avg_ms': 16.321420000058424,
 'median_ms': 14.43699999981618,
 'min_ms': 12.50369999979739,
 'max_ms': 100.83730000042124,
 'status_codes': {'200': 50}}

## Inférence batch vs inférence unitaire

L’inférence batch consiste à scorer plusieurs clients en une seule requête.

Elle permet de réduire les coûts répétés liés :
- aux appels Python ;
- aux conversions pandas / numpy ;
- à la validation scikit-learn ;
- au prétraitement répété.

Dans ce projet, le batching constitue l’optimisation la plus efficace.

In [4]:
batching

{'single_total_ms_for_50_calls': 146.37039999979606,
 'single_avg_ms_per_row': 2.9274079999959213,
 'batch_total_ms_for_50_rows': 4.449999998541898,
 'batch_avg_ms_per_row': 0.08899999997083796,
 'speedup_factor_per_row': 32.89222472983287}

## Analyse des goulots d’étranglement avec cProfile

Le profiling montre que les principaux coûts proviennent de :
- la validation scikit-learn ;
- l’imputation ;
- les conversions entre pandas et numpy ;
- le prétraitement ;
- l’appel LightGBM.

Ces résultats indiquent que le modèle lui-même n’est pas le seul facteur de latence : l’enrobage pipeline et la préparation des données jouent un rôle important.

## Test ONNX Runtime

Une conversion ONNX “best effort” a été testée sur le classifieur LightGBM après prétraitement Python.

L’objectif était d’évaluer si ONNX Runtime pouvait accélérer l’inférence du modèle.

Résultat :
- ONNX Runtime accélère l’inférence du classifieur ;
- le gain est réel mais modéré ;
- le prétraitement reste effectué en Python / scikit-learn.

Ainsi, ONNX est validé comme preuve de concept, mais le batching reste l’optimisation la plus impactante pour ce projet.

In [7]:
from pathlib import Path
import json
import pandas as pd

# Trouver automatiquement la racine du projet
ROOT = Path().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

perf_dir = ROOT / "reports" / "performance"

# Chargement des benchmarks
with open(perf_dir / "inference_benchmark.json", encoding="utf-8") as f:
    inference = json.load(f)

with open(perf_dir / "api_benchmark.json", encoding="utf-8") as f:
    api = json.load(f)

with open(perf_dir / "batching_benchmark.json", encoding="utf-8") as f:
    batching = json.load(f)

# ONNX (optionnel)
onnx_path = perf_dir / "onnx_benchmark.json"

if onnx_path.exists():
    with open(onnx_path, encoding="utf-8") as f:
        onnx = json.load(f)
else:
    onnx = {
        "status": "missing",
        "interpretation": "Benchmark ONNX non disponible."
    }

inference, api, batching, onnx

({'n_rows': 200,
  'n_loops': 20,
  'avg_ms': 11.654050000197458,
  'min_ms': 10.858900001039729,
  'max_ms': 18.788399998811656,
  'stats_file': 'reports\\performance\\cprofile_inference.prof',
  'top20_file': 'reports\\performance\\cprofile_inference_top20.txt'},
 {'n_requests': 50,
  'avg_ms': 16.321420000058424,
  'median_ms': 14.43699999981618,
  'min_ms': 12.50369999979739,
  'max_ms': 100.83730000042124,
  'status_codes': {'200': 50}},
 {'single_total_ms_for_50_calls': 146.37039999979606,
  'single_avg_ms_per_row': 2.9274079999959213,
  'batch_total_ms_for_50_rows': 4.449999998541898,
  'batch_avg_ms_per_row': 0.08899999997083796,
  'speedup_factor_per_row': 32.89222472983287},
 {'status': 'success',
  'pipeline_path': 'G:\\Mon Drive\\OC\\Projet_6\\credexp\\artifacts\\models\\pipeline.joblib',
  'holdout_path': 'G:\\Mon Drive\\OC\\Projet_6\\credexp\\data\\processed\\api_holdout.parquet',
  'onnx_model_path': 'G:\\Mon Drive\\OC\\Projet_6\\credexp\\artifacts\\models\\lightgbm_mode

In [8]:
summary = pd.DataFrame(
    [
        {
            "Expérience": "Pipeline scikit-learn local",
            "Métrique": "temps moyen / batch (ms)",
            "Valeur": inference.get("avg_ms"),
        },
        {
            "Expérience": "API FastAPI",
            "Métrique": "latence médiane (ms)",
            "Valeur": api.get("median_ms"),
        },
        {
            "Expérience": "Batching",
            "Métrique": "facteur d’accélération / ligne",
            "Valeur": batching.get("speedup_factor_per_row"),
        },
        {
            "Expérience": "ONNX Runtime",
            "Métrique": "facteur d’accélération",
            "Valeur": onnx.get("native_vs_onnx_speedup"),
        },
    ]
)

summary

,Expérience,Métrique,Valeur
0,Pipeline scikit-learn local,temps moyen / batch (ms),11.654050
1,API FastAPI,latence médiane (ms),14.437000
2,Batching,facteur d’accélération / ligne,32.892225
3,ONNX Runtime,facteur d’accélération,2.392050


In [9]:
status = onnx.get("status")

if status == "success":
    print("✅ Benchmark ONNX réussi")
    print(f"Temps pipeline natif total : {onnx.get('native_pipeline_total_ms'):.3f} ms")
    print(f"Temps ONNX total : {onnx.get('onnx_total_ms'):.3f} ms")
    print(f"Gain ONNX : x{onnx.get('native_vs_onnx_speedup'):.2f}")
elif status == "onnx_failed":
    print("⚠️ Conversion ou exécution ONNX échouée")
    print(onnx.get("onnx_error"))
    print(onnx.get("interpretation"))
else:
    print("Benchmark ONNX non disponible")

✅ Benchmark ONNX réussi
Temps pipeline natif total : 25.185 ms
Temps ONNX total : 10.529 ms
Gain ONNX : x2.39


## Conclusion

Le système est compatible avec un usage quasi temps réel.

Les résultats montrent :
- une faible latence API ;
- un coût d’inférence local maîtrisé ;
- un gain très important grâce au batching ;
- un gain complémentaire avec ONNX Runtime ;
- des goulots d’étranglement principalement liés au prétraitement et à la validation des données.

Configuration retenue :
- conserver FastAPI pour le serving ;
- conserver le pipeline scikit-learn pour la robustesse ;
- exposer `/predict` et `/predict_batch` ;
- privilégier `/predict_batch` lorsque plusieurs clients doivent être scorés ;
- surveiller la latence via Prometheus et Grafana.